<a href="https://colab.research.google.com/github/ecuirty-kr/1_DataAnalysis/blob/main/summary.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[구글 코랩(Colab)에서 실행하기](https://colab.research.google.com/github/lovedlim/bigdata_analyst_cert_v2/blob/main/part2/ch8/ch8_ex_regression.ipynb)

# Section1

### 베이스라인

# Section2

In [21]:
### 작업형 2 복기 : 원-핫 인코딩 concat() 합치기 [:len][len:] 분리 - 떨어지면 망임. 진짜 망.임
### 빅분기 실기 체험환경 확인 (완료) : 작업형 2번 풀어봄

## 문제조건 : 노트북 정보로 가격을 예측하시오
## 예측 컬럼 : price (가격 : 회귀)
## 평가지표 : R^2 (결정계수)

# 1. 라이브러리 및 데이터
import pandas as pd
train = pd.read_csv("https://raw.githubusercontent.com/lovedlim/bigdata_analyst_cert/main/part2/ch8/laptop_train.csv")
test = pd.read_csv("https://raw.githubusercontent.com/lovedlim/bigdata_analyst_cert/main/part2/ch8/laptop_test.csv")
# 데이터 확인
print(train.head(4), test.head(4))

# 2. 탐색적 데이터 분석 (크기 / 자료형 / 결측치 / 카테고리 / 기초통계량)
# print(train.shape, test.shape) # (91, 10) (39, 9)
# print(train.info(), test.info()) # float64(2), object(7) (train + int 1)
# print(train.isnull().sum(), test.isnull().sum())  # 결측치 다양하게 보유
# print(train['Price'].describe())

### 카테고리 동일 확인
# cols = train.select_dtypes(include='O').columns
# for col in cols:
#   set_train = set(train[col])
#   set_test = set(test[col])
#   same = (set_train==set_test)
#   if same:
#     print("카테 동일")
#   else:
#     print("동일 아님")  // 전부 다른 데이터

# 3. 데이터 전처리
target = train.pop('Price')

c_cols = ['Model', 'Series', 'Processor', 'Processor_Gen', 'Hard_Disk_Capacity', 'OS'] ## 범주형 결측치 처리 위한 컬럼명 추출
train[c_cols] = train[c_cols].fillna('X')
test[c_cols] = test[c_cols].fillna('X')
# 수치형 결측치 처리
n_cols = ['RAM']
train[n_cols] = train[n_cols].fillna(-1)
test[n_cols] = test[n_cols].fillna(-1)
# print(train.isnull().sum(), test.isnull().sum()) # 결측치 완전 처리

## 인코딩 (# 1 원-핫 기본)
# train = pd.get_dummies(train)
# test = pd.get_dummies(test)
# print(train.shape, test.shape) # (91, 97) (39, 74)

## 인코딩 (# 2 원-핫 합치기)
combined = pd.concat([train, test])  # 새 변수 combined : pandas 함수 .concat([train, test]) 같이
combined_dummies = pd.get_dummies(combined) # 새 변수에 / 위의 변수 combined 를 pd.get_dummies

n_train = len(train) # train 길이 변수
train = combined_dummies[:n_train] # 처음부터 train 길이까지 / combined get_dummies 한거에 적용
test = combined_dummies[n_train:] # train 길이부터 끝까지

## 인코딩 (# 3 레이블)
# from sklearn.preprocessing import LabelEncoder
# cols = train.select_dtypes(include='O').columns
# for col in cols:
#   le = LabelEncoder()
#   train[col] = le.fit_transform(train[col])
#   test[col] = le.transform(test[col])
  # *** 레이블 인코딩도 합쳐서 진행 가능

## 인코딩 (# 3 레이블 합치기) : 참고만
# combined = pd.concat([train,test])
# cols = train.select_dtypes(include='O').columns

# for col in cols:
#   le = LabelEncoder()
#   combined[col] = le.fit_transform(combined[col])
# n_train = len(train)
# train = combined[:n_train]
# test = combined[n_train:]

# 4. 검증 데이터 분리
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(train, target, test_size=0.2, random_state=0)
# print(y_train.shape, y_val.shape) # 시리즈 형태만 확인 (72,) (19,)

# 5. 모델 학습 및 평가(R^2)
## 랜포(회귀)
from sklearn.ensemble import RandomForestRegressor
rf = RandomForestRegressor(random_state=0)
rf.fit(X_train, y_train)
pred = rf.predict(X_val)
# 평가
from sklearn.metrics import r2_score #### RMSEL : root_mean_sqaured_log_error
r2 = r2_score(y_val, pred)
print("RF r2-score : ", r2) # 결과 :  RF r2-score :  0.7496764602229047

## LightGBM (회귀)
import lightgbm as lgb
lg = lgb.LGBMRegressor(random_state=0, verbose=-1)
lg.fit(X_train, y_train)
pred = lg.predict(X_val)
# 평가
lg_r2 = r2_score(y_val, pred)
print("GBM r2-score : ", lg_r2) # 결과 : GBM r2-score :  0.36199992800967373
## 성능 비교 : 더 작은 값 우수 RF > LGBM (LightGBM 사용)

# 6. 검증 및 결과 파일 생성
pred = lg.predict(test)
result = pd.DataFrame({'pred':pred})
result.to_csv("result.csv", index=False)

# 검증
print(pred.shape) # 결과 : (39,) 확인
file = pd.read_csv("result.csv")
file ## 확인

  Brand     Model Series Processor Processor_Gen   RAM Hard_Disk_Capacity  \
0  ASUS  VivoBook     15        i3          10th   8.0         512 GB SSD   
1  DELL  Inspiron    NaN        i3          11th   8.0           1 TB HDD   
2  ASUS  VivoBook     15        i7          10th  16.0         512 GB SSD   
3  DELL       NaN    NaN        i3          10th   8.0           1 TB HDD   

                OS  Rating  Price  
0  Windows 11 Home     4.3  37940  
1  Windows 11 Home     3.7  39040  
2  Windows 11 Home     4.1  57940  
3       Windows 10     3.2  41340       Brand    Model Series Processor Processor_Gen  RAM Hard_Disk_Capacity  \
0    DELL   Vostro    NaN        i3          10th  8.0         256 GB SSD   
1  Lenovo  IdeaPad      3        i3          10th  8.0         256 GB SSD   
2      HP      NaN    NaN        i5          11th  8.0         512 GB SSD   
3  Lenovo  IdeaPad      3        i3          11th  8.0         256 GB SSD   

                OS  Rating  
0  Windows 10 Home 

,pred
0,45439.861522
1,37544.283389
2,68932.752098
3,37557.853742
4,53361.660510
5,47378.857341
6,70379.622671
7,68315.465051
8,70379.622671
9,53634.403771


### 베이스라인

In [23]:
###### 이진/다중 분류 평가지표  (정확도 / 정밀도 / 민감도(=재현율) / 오류율 (?))
### 1. 정확도 라이브러리
from sklearn.metrics import accuracy_score

### 2. 정밀도 라이브러리
from sklearn.metrics import precision_score ## 얘는 잘 안 나옴

### 3. 재현율(민감도) 라이브러리
from sklearn.metrics import recall_score


###### 회귀 평가지표 (MSE / MAE / 결정계수(R^2) / RMSE / MSLE / RMSLE / MAPE)
### 1. MSE 라이브러리
from sklearn.metrics import mean_squared_error

### 2. MAE 라이브러리
from sklearn.metrics import mean_absolute_error

### 3. 결정계수(R^2) 라이브러리
from sklearn.metrics import r2_score

### 4. RMSE 라이브러리
from sklearn.metrics import root_mean_squared_error

### 5. MSLE 라이브러리
from sklearn.metrics import mean_squared_log_error ## l = log

### 6. RMSLE 라이브러리
from sklearn.metrics import root_mean_squared_log_error

### 7. MAPE 라이브러리
from sklearn.metrics import mean_absolute_percentage_error ## p = percentage(%)

### 성능개선

In [33]:
##### 단일 표본 검정 / 대응 표본 검정 / 독립 표본 검정
## 1. 단일 표본 검정 : 특정 집단의 평균이 특정 값과 유의미하게 다른지  (ex. 팝콘 무게 / 120 g)
df = pd.read_csv("https://raw.githubusercontent.com/lovedlim/bae-v3/main/part4/ch10/subject_performance.csv")

from scipy.stats import ttest_1samp
t_statistic, p_value = ttest_1samp(df['student_id'], 120) ## 대립 가설이 120 보다 작다, 면 alternative='less' 옵션 추가(단측) - alternative='greater' : 크다


## 2. 대응 표본 검정 : 동일한 그룹에 대해 시간차를 두고 측정 결과 비교 (ex. 약 효과, 교육 프로그램 성과)
from scipy.stats import ttest_rel
print(ttest_rel(df['before'], df['after'], alternative='less')) ## 대립 : 새로운 교육 프로그램이 효과가 크다 (before-after 값 < 0 이면 'less' / > 0 이면 'greater')


## 3. 독립 표본 검정 : 서로 다른 두 그룹 간의 평균이 서로 다름을 판단  (ex. 1, 2 반의 수학 점수 평균)
from scipy.stats import ttest_ind
t_statistic, p_value = ttest_ind(class1, class2, equal_val=True, alternative='greater') ## c1 점수 평균이 더 높다 (c1 - c2 > 0 "greater") / equal_val (모분산 동일)


###### 분산 분석 (일원 분산 분석 / 이원 분산 분석)
### 1. 일원 분산 분석 : 3 가지 이상의 그룹 간의 평균 차이가 통계적으로 유의한지  (ex. 4 종류의 비료 데이터)
from scipy.stats import f_oneway
t_statistic, p_value = f_oneway(df['A'], df['B'], df['C'], df['D']) ### 일원 = f_oneway

from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm
model = ols('종속~독립',data=df).fit()
anova_lm(model)

### 2. 이원 분산 분석 : 요인의 수가 2개인 경우   (ex. 학습 방법 및 학습 장소에 따른 시험성적)
import statsmodels.api as sm
anova_table = sm.stats.anova_lm(model)


######## 카이제곱 : 적합도 검정 / 독립성 검정 / 동질성 검정
## 1. 적합도 검정 : 1 개의 범주형 변수가 특정 분포(전국적 조사)를 잘 따르고 있는지  (ex. 아이스크림 맛 선호도)
from scipy.stats import chisquare
# 관측 빈도 / 예측 빈도
observed = [150, 120, 30] # 관측 빈도 : 수
expected = [0.5*300, 0.35*300, 0.15*300] # 기대 빈도 : 비율   ->  빈도로 변환(비율 * 전체 수)
chisquare(observed, expected)

## 2. 독립성 검정 : 2 개의 변수가 서로 독립적인지 연관인지  (ex. 성별에 따른 운동 선호도)
# 1. 교차표  2. 로우 데이터
from scipy.stats import chi2_contingency
# df = [[80,30], [90,10]] # 데이터 프레임 만들기 (교차표 데이터)
chi2_contingency(df)

# data = {
#   '성별' : ['남자']*110 + ['여자']*100,
#   '운동' : ['좋아함']*80 + ['안좋']*30 + ['좋아함']*90 + ['안좋']*10
# }
# df = pd.DataFrame(data)
df = pd.crosstab(df['성별'], df['운동']) # 교차표 데이터 변환
chi2_contingency(df)

## 3. 동질성 검정 : 2 개 이상의 집단에서 분산의 동질성을 갖는지 검정
from scipy.stats import chi2_contingency
# df = pd.DataFrame([[50,50],[30,70]]) # 교차표 데이터
chi2_contingency(df)

# 로우 데이터 (data->DataFrame(data))
# data = {
#     '학과' : ['통계학과']*100 + ['컴퓨터공학과']*100,
#     '가입여부' : ['가입']*50 + ['미가입']*50 + ['가입']*30 + ['미가입']*70
# }
# df = pd.DataFrame(data)
chi2_contingency(df)

KeyError: 'before'

# Section3

In [36]:
##### 회귀 분석 (상관계수 / 결정 계수 / 회귀계수(기울기와 절편) / p-value / 잔차 제곱합 / 회귀모델 MSE)
## 1. 상관계수
value = df['키'].corr(df['몸무게']) # 두 변수 간의 상관계수 (기본: 피어슨 상관계수)

## 2. 결정계수
value = model.rsquared

## 3. 회귀계수
value = model.params['몸무게']

## 4. p-value
p_value = model.pvalues['몸무게']
print("{:.10f}".format(model.pvalues['몸무게'])) ## 지수 -> 실수 변환

## 5. 잔차제곱합 : 실제값 - 모델
df['잔차'] = df['키'] - model.predict(df)
print("잔차제곱합: ", sum(df['잔차']**2))

## 6. 회귀 모델 MSE (평균 제곱 오차)
df['잔차'] = df['키'] - model.predict(df)
MSE = (df['잔차']**2).mean()


## 단순 선형 회귀 분석
from statsmodels.formula.api import ols
model = ('종속~독립+독립', data=df).fit()


## 새 데이터로 회귀 모델 예측 수행
new_data = pd.DataFrame({
    '몸무게' : [67]
})
result = model.predict(new_data)
print(result[0]) # 값만 출력


############ 신뢰구간 95%
print(model.conf_int(alpha=0.05).loc['몸무게'])  ## model.conf_int(alpha=0.05) -> 신뢰구간

############ 예측 키의 신뢰 구간과 예측 구간 (몸무게가 50 일 때)
new_data = pd.DataFrame({ '몸무게' : [50]})
pred = model.get_prediction(new_data)
result = pred.summay_frame(alpha=0.05) ## 95% 신뢰구간 데이터 (상한/하한)   ** summay_frame(alpha=0.05) 옵션 값으로 alpha=신뢰구간 들어감
print(result)

SyntaxError: invalid syntax. Maybe you meant '==' or ':=' instead of '='? (1242762470.py, line 26)

In [ ]:
### 다중 선형 회귀 (상관 계수 / t-검정의 p-value / 결정 계수 / 회귀 계수 / p-value / 잔차제곱합)
# 1. 상관계수
df['광고비'].corr(df['매출액'])


# 2. t-검정의 p-value
from scipy import stats
stats.pearsonr(df['광고비'],df['매출액'])  ## t-검정의 p-value 는 stats.pearsonr(df['1'], df['2'])

# 3. 결정계수
model.rsquared

# 4. 회귀 계수  #### summary() 했을 때 coef 값 = 회귀계수
model.params

# 5. p-value
model.pvalues

# 6. 잔차 제곱합
df['잔차'] = df['종속변수'] - model.predict(df)
sum(df['잔차']**2)

### 베이스라인

In [37]:
#### 오즈비
import numpy as np
np.exp(model.params['bmi']*3) ## 증가 단위수 만큼 model.params['변수'] (변수의 회귀계수)에 곱하면 구할 수 있음


#### 대비 오즈비 (ex. 'A'인 사람 대비 'S'인 사람의 오즈비)
np.exp(model.params['lifestyle[T.S]'])  # summary() 참고

## 데이터 분할 (처음 700행 : 300행)
df = pd.read_csv('file.csv')
train = df.iloc[:700]
test = df.iloc[700:] # 데이터 크기 확인

## 복기 (기본)

# 4분위수
df['변수'].quantile(0.25) # 1 분위수
df['변수'].quantile(0.75) # 3 분위수

NameError: name 'model' is not defined

### 성능개선